# Symmetric and Asymmetric Encryption

### Symmetric

In [27]:
from cryptography.fernet import Fernet
import random

# 1. Generate the Symmetric Key
key = Fernet.generate_key()

# Initialize the Fernet cipher with your key
cipher = Fernet(key)

# --- The Encryption/Decryption Process ---
message = b"Hallo Hier"

# 2. Encrypt the message
ciphertext = cipher.encrypt(message)

print(f"The Key (keep secret!): {key.decode('utf-8')}")
print(f"Encrypted token: {ciphertext[:40]}... (truncated)")

# 3. Decrypt the message
plaintext = cipher.decrypt(ciphertext)

print(f"Decrypted: {plaintext.decode('utf-8')}")

The Key (keep secret!): a_4np6-00Pt4bSCebQZLx7bnER7BAkSj87FnYyGiCjo=
Encrypted token: b'gAAAAABpsCOMEY9MLzF9n13MDnI80v_AxGiVxlBe'... (truncated)
Decrypted: Hallo Hier


### Asymmetric

In [28]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes

# 1. Generate the Private Key
# 65537 is the standard public exponent. 2048 is a secure, standard key size.
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)

# 2. Extract the Public Key (to "share" with whoever is sending the message)
public_key = private_key.public_key()

# --- The Encryption/Decryption Process ---

message = b"Hallo Hier"

# 3. Encrypt the message using the PUBLIC key
# OAEP padding with SHA256 is the modern, secure standard for RSA encryption.
ciphertext = public_key.encrypt(
    message,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(f"Encrypted (raw bytes): {ciphertext[:20]}... (truncated)")

# 4. Decrypt the message using the PRIVATE key
plaintext = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(f"Decrypted: {plaintext.decode('utf-8')}")

Encrypted (raw bytes): b"61u\x84\xb5\xed'\xdbN\x06\xf9U\x9f\xf8/\xcc{\xd9\xda\x00"... (truncated)
Decrypted: Hallo Hier


# Authentication

In [29]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes
import random

PADDING = padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )

SIGN_PADDING = padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    )

class Party:
    def __init__(self) -> None:
        self.privateKey = rsa.generate_private_key(public_exponent=65537, key_size=2048)
        self.publickey = self.privateKey.public_key()
        self.nonce = None
        pass
    
    def generate_R(self):
        self.nonce = random.randint(0, int(10e4))
        return self.nonce
    
    def encrypt_message(self, encryption_key, input_text):
        return encryption_key.encrypt(input_text, PADDING)
    
    def decrypt_message(self, decryption_key, input_text):
        return decryption_key.decrypt(input_text, PADDING)
    
    def sign_message(self, input_text):
        return self.privateKey.sign(input_text, SIGN_PADDING, hashes.SHA256())
    
    def verify_signature(self, public_key, signature, original_text):
        try:
            public_key.verify(signature, original_text, SIGN_PADDING, hashes.SHA256())
            return True
        except Exception:
            return False

1. A → B: I want to communicate
2. A ← B: Send KT+
3. A → B: KT+
4. A ← B: R
5. A → B: KT-(R)
6. B verifies: KT+(KT-(R)) =? R

In [30]:
A = Party()
B = Party()

# 1. A → B: I want to communicate
print("STEP 1: A says: I want to communicate")

# 2. A ← B: Send KT+
print("STEP 2: B says: Send me your public key (KT+)")

# 3. A → B: KT+
print(f"STEP 3: A sends public key (KT+): {A.publickey}")

# 4. A ← B: R
R = B.generate_R()
R_bytes = str(R).encode()
print(f"STEP 4: B sends nonce R: {R}")

# 5. A → B: KT-(R)  (A signs R with private key)
signature = A.sign_message(R_bytes)
print(f"STEP 5: A sends KT-(R) (signature): {signature[:20]}... (truncated)")

# 6. B verifies: KT+(KT-(R)) =? R
verified = B.verify_signature(A.publickey, signature, R_bytes)
print(f"STEP 6: B verifies KT+(KT-(R)) == R: {verified}")
print(f"RESULT: Authentication {'SUCCESSFUL' if verified else 'FAILED'}")

STEP 1: A says: I want to communicate
STEP 2: B says: Send me your public key (KT+)
STEP 3: A sends public key (KT+): <cryptography.hazmat.bindings._rust.openssl.rsa.RSAPublicKey object at 0x0000019BAF13FFD0>
STEP 4: B sends nonce R: 6948
STEP 5: A sends KT-(R) (signature): b'\xb6\xd8\xb9\xefDB\xdc\xf7\xb7c\x8e3y\xc2\xcb\x1c\xebG-\x14'... (truncated)
STEP 6: B verifies KT+(KT-(R)) == R: True
RESULT: Authentication SUCCESSFUL
